# Processing LIDAR Data. 
This project revolves around processing the lidar data. It is stored in anscii or bit data. Those have been converted to CSV or TXT  files with meta data. I need to process those to usable formats. So trying to put them into a larger pandas dataframe with real world x, y, z and backscatter values. This notebook will try to go through that. 

### Imports and config
Using the typical imports, I can probably trim this down in the future. AND I want to start using my config files so the initial commands to have that working. 

In [26]:
import pandas as pd
import numpy as np
import math
import datetime as dt
from matplotlib import pyplot as plt

import os
import shutil
import glob

from netCDF4 import Dataset
import xarray as xr

from scipy.interpolate import griddata
from scipy.signal import argrelextrema, find_peaks

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent / "src"))

from config import load_config

I can call the directories from the config file. This makes it easier to update directories if ever I need to...

In [27]:
# test loading config 
paths = load_config('paths.yml')
paths['paths']['usc_interim_dir']

'C:/Users/ben/OneDrive - University of South Carolina/SAVANT_Data/01_interim/USC'

### Organizing basic instrument configuration information. 
For now I will create a dictionary, but in the future I will put this in a config file, but for now this is fine.

In [12]:
# creating a library of lidar locations
lidars = {
    'USC Lidar': {
        'bin_width': 7.5,
        'lidar_height': 2,
        'X': 293677.5,  # 293701.1 # 293705.8   #293705.7868 # 293677.5, 393553.7
        'Y': 393553.7,  # 393557.9 #393522.7   #393522.7423
        'Z': 337,  # 236.8 #235.5      #235.4768
        'Azimuth': 118,  # 120 #110
        'Zenith': 0
        },
    'UIUC Lidar': {
        'bin_width': 3.5,
        'lidar_height': 2,
        'X': 293946.1,   #293946.1075
        'Y': 393318.1,   #393318.1287
        'Z': 233.6,      #233.589
        'Azimuth': 296,
        'Zenith': 0
        }
        }

I can call these values when I need to...

In [ ]:
lidars['UIUC Lidar']

{'bin_width': 3.5,
 'lidar_height': 2,
 'X': 293946.1,
 'Y': 393318.1,
 'Z': 233.6,
 'Azimuth': 296,
 'Zenith': 0}

### Setting up directories where I need them.
Most raw or intermediate data will be stored in OneDrive.


In [28]:
lidar_data_dir = paths['paths']['lidar_dir']